In [1]:
# ============================================================
# CONFIGURAÇÃO INICIAL DO AMBIENTE
# ============================================================
#
# Este notebook constrói a camada Bronze da pipeline.
#
# Nesta célula, configuramos:
#
#   - raiz do projeto;
#   - diretório local dos arquivos extraídos;
#   - interpretador Python usado pelo PySpark;
#   - credenciais temporárias da AWS;
#   - bucket S3 do integrante.
#
# Cada integrante deve executar o notebook com seu próprio arquivo .env,
# suas próprias credenciais do AWS Academy Learner Lab e seu próprio
# bucket S3.
#
# Pontos de configuração esperados no .env:
#
#   AWS_ACCESS_KEY_ID      Chave de acesso temporária da AWS.
#   AWS_SECRET_ACCESS_KEY  Chave secreta temporária da AWS.
#   AWS_SESSION_TOKEN      Token temporário da sessão AWS Academy.
#   S3_BUCKET              Nome do bucket S3 do integrante.
#
# ============================================================
# BIBLIOTECAS
# ============================================================

# Setup Env: Dotenv/Pathlib
import os, sys
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

# ============================================================
# CAMINHOS DO PROJETO E VARIÁVEIS DE AMBIENTE
# ============================================================

# Localiza o arquivo .env, carrega suas variáveis e define a raiz do projeto.
# O diretório EXTRAIDOS_DIR contém os arquivos gerados no notebook
# 00_aquisicao_dados.ipynb.
load_dotenv(find_dotenv())
PROJECT_ROOT = Path(find_dotenv()).parent

# ============================================================
# CONFIGURAÇÃO DO PYSPARK
# ============================================================

# Garante que o PySpark use o mesmo interpretador Python do ambiente atual.
# Isso evita conflitos comuns em WSL, Conda ou ambientes virtuais.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# ============================================================
# CREDENCIAIS AWS E BUCKET S3
# ============================================================

# Carrega as credenciais temporárias e o bucket S3 definidos no .env.
AWS_ACCESS_KEY_ID     = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN     = os.getenv("AWS_SESSION_TOKEN")
S3_BUCKET             = os.getenv("S3_BUCKET")

# Interrompe a execução se alguma configuração obrigatória não foi carregada.
if not all([AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_SESSION_TOKEN, S3_BUCKET]):
    raise EnvironmentError("Falha ao carregar credenciais AWS ou S3_BUCKET do arquivo .env")

# ============================================================
# CONFERÊNCIA DA CONFIGURAÇÃO
# ============================================================

print(f"Projeto: {PROJECT_ROOT}\nBucket : {S3_BUCKET}")

Projeto: /mnt/d/diego/01_projects/postech-challenge-2
Bucket : alfabetizacao-data-lake-diego


In [2]:
# ============================================================
# CRIAÇÃO DA SESSÃO SPARK COM ACESSO AO S3
# ============================================================
#
# Esta célula cria a SparkSession usada para processar os arquivos
# locais extraídos e gravar os resultados da camada Bronze no S3.
#
# A sessão é executada localmente com master("local[*]"), usando os
# núcleos disponíveis da máquina.
#
# Pacotes configurados:
#
#   hadoop-aws
#       Permite acesso ao S3 via protocolo s3a://.
#
#   aws-java-sdk-bundle
#       Fornece dependências da AWS usadas pelo conector S3A.
#
#   spark-excel
#       Permite leitura de arquivos XLSX com Spark.
#
# Como o AWS Academy Learner Lab usa credenciais temporárias, a sessão
# Spark utiliza TemporaryAWSCredentialsProvider e recebe também o
# AWS_SESSION_TOKEN.
#
# Entrada:
#
#   AWS_ACCESS_KEY_ID
#   AWS_SECRET_ACCESS_KEY
#   AWS_SESSION_TOKEN
#
# Saída:
#
#   spark  Sessão Spark configurada.
#
# ============================================================
# BIBLIOTECAS
# ============================================================

# SparkSession: S3 Config
from pyspark.sql import SparkSession

# ============================================================
# SESSÃO SPARK
# ============================================================

spark = (
    SparkSession.builder
    .appName("exploracao-e-armazenar-S3")
    .master("local[*]")
    # Configuração de Pacotes e S3A
    .config("spark.jars.packages","org.apache.hadoop:hadoop-aws:3.3.4,""com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.access.key", AWS_ACCESS_KEY_ID)
    .config("spark.hadoop.fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.session.token", AWS_SESSION_TOKEN)
    .config("spark.hadoop.fs.s3a.endpoint", "s3.amazonaws.com")
    .getOrCreate()
)

# ============================================================
# CONFERÊNCIA DA SESSÃO
# ============================================================

print(f"Sessão Spark {spark.version} criada com sucesso.")

26/07/06 17:56:59 WARN Utils: Your hostname, lua resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/06 17:56:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/diego/miniconda3/envs/postech2/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/diego/.ivy2/cache
The jars for the packages stored in: /home/diego/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-319c2114-0d79-45ef-8215-2cf1d85386ed;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 295ms :: artifacts dl 13ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evict

Sessão Spark 3.5.5 criada com sucesso.


In [3]:
# ============================================================
# CLIENTE S3 PARA OPERAÇÕES ADMINISTRATIVAS
# ============================================================
#
# Esta célula cria um cliente S3 com boto3 usando as mesmas
# credenciais temporárias carregadas do arquivo .env.
#
# O Spark será usado para processar e gravar os dados da camada
# Bronze. O boto3 fica reservado para operações administrativas
# no bucket, como listar, validar ou limpar prefixos quando necessário.
#
# Entrada:
#
#   AWS_ACCESS_KEY_ID
#   AWS_SECRET_ACCESS_KEY
#   AWS_SESSION_TOKEN
#
# Saída:
#
#   s3_client  Cliente boto3 configurado para acessar o S3.
#
# ============================================================
# BIBLIOTECAS
# ============================================================

# Boto3: Gestão de pastas
import boto3

# ============================================================
# CLIENTE S3
# ============================================================

# Inicializa o cliente S3 logo após o setup das credenciais.
s3_client = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN
)

In [4]:
from pyspark.sql import functions as F

BRONZE_BASE  = f"s3a://{S3_BUCKET}/bronze"   # Bronze no S3
SILVER_BASE = f"s3a://{S3_BUCKET}/silver"   # sILVER no S3
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

# A DIAGONAL: coluna da taxa do ano corrente, por publicação.
TAXA_CORRENTE = {2023: "PC_ALUNO_ALFABETIZADO",
                 2024: "PC_ALUNO_ALFABETIZADO_2024",
                 2025: "PC_ALUNO_ALFABETIZADO_2025"}

def taxa_diagonal():
    # CASE por ano: cada linha pega a taxa da SUA publicação.
    # mergeSchema une as 3 colunas; em cada partição só a do ano corrente é não-nula.
    itens = list(TAXA_CORRENTE.items())
    expr = F.when(F.col("NU_ANO_AVALIACAO") == itens[0][0], F.col(itens[0][1]))
    for ano, col in itens[1:]:
        expr = expr.when(F.col("NU_ANO_AVALIACAO") == ano, F.col(col))
    return expr

def normaliza_meta(colname):
    # metas do UF são string: '>80'/'> 80' -> 80 ; '- ' -> null ; número -> double
    c = F.trim(F.col(colname))
    return F.when(c.startswith(">"), F.lit(80.0)).otherwise(c.cast("double"))

In [5]:
uf_bronze = spark.read.option("mergeSchema", "true").parquet(f"{BRONZE_BASE}/metas_ufs")

brasil_silver = (uf_bronze
    .filter(F.col("NOME_UF") == "Brasil")
    .select(
        F.col("NU_ANO_AVALIACAO").alias("ano"),
        F.initcap(F.col("REDE")).alias("rede"),                     # PÚBLICA -> Pública
        taxa_diagonal().alias("taxa_alfabetizacao"),
        *[normaliza_meta(f"META_FINAL_{a}").alias(f"meta_alfabetizacao_{a}")
          for a in range(2024, 2031)],
        F.col("PC_AVALIADOS_LP").alias("percentual_participacao"),
    ))

(brasil_silver.write.mode("overwrite").partitionBy("ano")
    .parquet(f"{SILVER_BASE}/meta_alfabetizacao_brasil"))

print("meta_alfabetizacao_brasil gravada.")
brasil_silver.orderBy(F.col("ano").desc()).show(truncate=False)

26/07/06 07:31:13 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


meta_alfabetizacao_brasil gravada.


+----+-------+------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+
|ano |rede   |taxa_alfabetizacao|meta_alfabetizacao_2024|meta_alfabetizacao_2025|meta_alfabetizacao_2026|meta_alfabetizacao_2027|meta_alfabetizacao_2028|meta_alfabetizacao_2029|meta_alfabetizacao_2030|percentual_participacao|
+----+-------+------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+
|2025|Pública|66.0              |60.0                   |64.0                   |67.0                   |71.0                   |74.0                   |77.0                   |80.0                   |89.0                   |
|2024|Pública|59.2              |59.9                   |63.77                  |67.47          